# Streamlit App Generation with DemoGPT

DemoGPT can automatically generate complete Streamlit applications from natural language instructions through a multi-stage pipeline. Instead of manually writing UI code, defining layouts, and wiring up LLM calls, you simply describe what you want the app to do in plain English, and DemoGPT produces a fully functional Streamlit application ready to run.

This notebook walks through the generation pipeline, demonstrates how to generate different types of apps, and covers configuration options.

## Setting Up Your API Key

Create a `.env` file in your project root with your OpenAI API key:

```
OPENAI_API_KEY=sk-your-key-here
```

Then load it with `python-dotenv` (already included with DemoGPT):

In [ ]:
%pip install demogpt

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")  # Loads OPENAI_API_KEY from .env file

import os
print("API key loaded:", "OPENAI_API_KEY" in os.environ)

## The Generation Pipeline

DemoGPT generates Streamlit apps through a **five-stage pipeline**. Each stage yields progress updates as a dictionary containing:

- **`stage`**: The name of the current stage
- **`percentage`**: Completion percentage (0-100)
- **`message`**: A human-readable status message
- **`done`**: Boolean indicating whether generation is complete

The stages are:

1. **`system_inputs`** - Detects what inputs the app needs (API keys, etc.)
2. **`plan`** - Creates a structured execution plan from the instruction
3. **`task`** - Breaks the plan into concrete executable tasks
4. **`draft`** - Converts each task into Python code snippets
5. **`final`** - Assembles all snippets into a complete Streamlit application

```
system_inputs -> plan -> task -> draft -> final
```

When the pipeline completes, the final yielded dictionary includes a `"code"` key containing the full Streamlit application source code.

## Basic App Generation

The simplest way to generate an app is to instantiate `DemoGPT`, provide an instruction and title, and iterate over the generation phases.

In [ ]:
from demogpt import DemoGPT

demo = DemoGPT(model_name="gpt-4o-mini")

instruction = "Create an app that takes a topic from the user and generates a summary about it"
title = "Topic Summarizer"

for phase in demo(instruction=instruction, title=title):
    print(f"[{phase['stage']}] {phase['percentage']}% - {phase['message']}")
    if phase["done"]:
        print("\n=== Generated Streamlit Code ===")
        print(phase["code"])

## Understanding the Stages

Each stage in the pipeline serves a specific purpose:

### 1. `system_inputs`
Analyzes the instruction to detect what system-level inputs the app requires. This includes API keys, environment variables, or other configuration that must be provided before the app can function.

### 2. `plan`
Takes the natural language instruction and creates a structured, high-level execution plan. This plan outlines the logical flow of the application -- what data comes in, how it gets processed, and what goes out.

### 3. `task`
Breaks the high-level plan into concrete, executable tasks. DemoGPT has **14 task types** available and selects the appropriate ones for the given plan. Each task maps to a specific operation like collecting user input, calling an LLM, or displaying output.

### 4. `draft`
Converts each task into actual Python code snippets. Each task type has associated code templates and generation logic that produces working Streamlit and LangChain code.

### 5. `final`
Assembles all the individual code snippets into a single, complete Streamlit application. This stage handles imports, variable wiring between components, and ensures the final script is coherent and runnable.

## Available Task Types

DemoGPT supports **14 task types** that can be composed to build a wide variety of applications:

| Task Type | Description |
|---|---|
| `ui_input_text` | Collects text input from the user via a Streamlit text field |
| `ui_input_file` | Provides a file uploader widget for document uploads |
| `ui_input_chat` | Creates a chat input interface for conversational apps |
| `ui_output_text` | Displays text output to the user |
| `ui_output_chat` | Renders chat messages in a conversational format |
| `prompt_template` | Constructs an LLM prompt from a template with variable substitution |
| `chat` | Manages a chat conversation with an LLM, maintaining message history |
| `doc_loader` | Loads documents from uploaded files (PDF, TXT, etc.) |
| `doc_to_string` | Converts loaded document objects into plain text strings |
| `string_to_doc` | Converts plain text strings into document objects |
| `python` | Executes custom Python logic for transformations or computations |
| `plan_and_execute` | Uses a plan-and-execute agent for complex multi-step reasoning |
| `summarization` | Summarizes long documents or text using LLM-based summarization chains |
| `question_answering` | Answers questions based on provided context documents |

The pipeline automatically selects which task types to use based on your instruction.

## Chat-based App Generation

DemoGPT can generate conversational applications that use chat input/output task types with message history management.

In [ ]:
instruction = "Create a chatbot that acts as a helpful coding assistant"
title = "Code Assistant Bot"

for phase in demo(instruction=instruction, title=title):
    print(f"[{phase['stage']}] {phase['percentage']}%")
    if phase["done"]:
        print("\n=== Generated Chat App Code ===")
        print(phase["code"])

## Document-based App

Generate an app that lets users upload documents and interact with their content. This example creates a PDF question-answering application using the `doc_loader`, `doc_to_string`, and `question_answering` task types.

In [ ]:
instruction = "Create an app where users can upload a PDF file and ask questions about its content"
title = "PDF Q&A App"

for phase in demo(instruction=instruction, title=title):
    if phase["done"]:
        print(phase["code"])

## Model Selection

You can choose which model to use at initialization and switch models later with `setModel()`.

In [ ]:
# Initialize with gpt-4o-mini
demo = DemoGPT(model_name="gpt-4o-mini")

# Switch to GPT-4o for better quality
demo.setModel("gpt-4o")

## Configuration Options

DemoGPT accepts several configuration parameters:

- **`max_steps`**: Maximum number of refinement iterations for code generation (default: 10). Increase this if the pipeline needs more attempts to produce valid code.
- **`plan_max_steps`**: Maximum number of refinement iterations for the planning stage (default: 3). Increase for more complex applications.
- **`openai_api_base`**: Custom API base URL for OpenAI-compatible endpoints.

In [ ]:
demo = DemoGPT(
    model_name="gpt-4o",
    max_steps=15,
    plan_max_steps=5
)

## Handling Errors

The generation pipeline may fail at any stage. Check for the `"failed"` key in each phase to detect and handle failures gracefully.

In [ ]:
for phase in demo(instruction="Create a translator", title="Translator"):
    if phase.get("failed"):
        print(f"Generation failed at stage: {phase['stage']}")
        break
    if phase["done"]:
        print("Success!")
        print(phase["code"])

## Running the Generated App

Once DemoGPT produces the Streamlit code, you have two options to run it:

### Option 1: Save and run manually

Save the generated code to a `.py` file and launch it with Streamlit:

```bash
streamlit run app.py
```

### Option 2: Use the DemoGPT CLI

Run the `demogpt` command to launch the built-in web interface, which provides an interactive UI for generating and previewing apps:

```bash
demogpt
```

This opens a Streamlit-based interface where you can enter instructions, watch the generation pipeline progress, and immediately preview the resulting application.

## Summary

In this notebook, we covered how DemoGPT generates complete Streamlit applications from natural language:

- **The five-stage pipeline**: `system_inputs` -> `plan` -> `task` -> `draft` -> `final`, each yielding progress updates as the generation proceeds.
- **Basic generation**: Instantiate `DemoGPT`, call it with an instruction and title, and iterate over the phases to get the final code.
- **14 task types**: From simple text input/output to document loading, chat interfaces, summarization, and question answering -- DemoGPT composes these automatically.
- **Different app types**: Text-based apps, chat-based apps, and document-based apps can all be generated from plain English descriptions.
- **Configuration**: Control model selection, refinement iterations, and API endpoints to tune the generation process.
- **Error handling**: Check for the `"failed"` key to gracefully handle generation failures.

DemoGPT turns the process of building LLM-powered Streamlit applications from hours of coding into a single function call.